<a href="https://colab.research.google.com/github/burcutatli/clinical-safe-rag/blob/main/clinical_safe_rag_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q chromadb anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [3]:
"""
MetaboLift Clinical-Safe Customer Support RAG Demo
===================================================
A Retrieval-Augmented Generation pipeline with clinical-safety routing.

Architecture: Path B (Modified)
  User question → Haiku classifier → {clinical: direct escalate}
                                   → {else: Chroma RAG → Sonnet answer}

Built as a proof-of-concept for a telehealth customer support scenario
where clinical questions must never be handled by the general LLM.

Author: Burcu Tatli
Stack:  Claude API (Sonnet 4.5 + Haiku 4.5), ChromaDB, Python
"""

from anthropic import Anthropic
from google.colab import userdata
import chromadb

# -----------------------------------------------------------------------------
# 1. SETUP
# -----------------------------------------------------------------------------
client = Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# -----------------------------------------------------------------------------
# 2. KNOWLEDGE BASE
# Synthetic MetaboLift support documents. In production these would come
# from Notion, Intercom Help Center, or an internal KB system.
# -----------------------------------------------------------------------------
documents = [
    {
        "id": "shipping",
        "text": (
            "MetaboLift ships all orders within 24 hours of approval. "
            "Delivery takes 3-5 business days via USPS Priority Mail. "
            "Tracking info is sent to your email. Free shipping on all orders. "
            "Discreet packaging with no MetaboLift branding on the outside."
        ),
    },
    {
        "id": "refunds",
        "text": (
            "MetaboLift offers a 7-day refund policy on unopened products. "
            "Refunds are processed within 7-10 business days after we receive "
            "the returned item. Opened medications cannot be refunded per FDA "
            "regulations. Contact support@metabolift.com for return authorization."
        ),
    },
    {
        "id": "semaglutide",
        "text": (
            "Semaglutide is a weekly injection taken on the same day each week. "
            "Start dose is typically 0.25mg for the first 4 weeks, then 0.5mg. "
            "Store in refrigerator. Common side effects include nausea and "
            "reduced appetite during the first month."
        ),
    },
    {
        "id": "texas_regulations",
        "text": (
            "In Texas, compounded semaglutide requires a valid prescription "
            "from a Texas-licensed physician. Texas State Board of Pharmacy "
            "oversees all compounding pharmacies. Patients must be 18+ and "
            "cannot have a history of medullary thyroid carcinoma. Telehealth "
            "consultations are accepted."
        ),
    },
    {
        "id": "billing",
        "text": (
            "MetaboLift subscriptions renew automatically each month on your "
            "anniversary date. You can pause or cancel anytime in your account "
            "dashboard. We accept Visa, MasterCard, HSA, and FSA cards. "
            "No long-term contracts."
        ),
    },
]

# -----------------------------------------------------------------------------
# 3. VECTOR DATABASE
# Chroma (in-memory). Auto-embeds documents with its default model
# (all-MiniLM-L6-v2). Free, fast, suitable for demos and small KBs.
# -----------------------------------------------------------------------------
chroma_client = chromadb.Client()

# Avoid collision if the cell is re-run in the same session
try:
    chroma_client.delete_collection(name="metabolift_kb")
except Exception:
    pass

collection = chroma_client.create_collection(name="metabolift_kb")
collection.add(
    documents=[d["text"] for d in documents],
    ids=[d["id"] for d in documents],
)

# -----------------------------------------------------------------------------
# 4. ROUTING LAYER
# A cheap, fast Haiku call classifies every incoming question.
# Clinical questions are hard-blocked before they reach the main LLM or RAG.
# This is an architectural guardrail, not a prompt-level one.
# -----------------------------------------------------------------------------
def classify_query(user_question: str) -> str:
    """Classify a customer question into one of five categories using Haiku."""
    result = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=20,
        system=(
            "You classify customer support questions into ONE category.\n"
            "Categories: shipping, billing, account, clinical, general\n\n"
            "CLINICAL means anything about:\n"
            "- Medication dosage or timing adjustments\n"
            "- Side effects someone is EXPERIENCING (not general questions)\n"
            "- Whether to stop/start/change medication\n"
            "- Medical symptoms or reactions\n"
            "- Drug interactions\n"
            "- 'Should I...' questions about health\n\n"
            "Respond with ONLY the category name, nothing else."
        ),
        messages=[{"role": "user", "content": user_question}],
    )
    return result.content[0].text.strip().lower()


# -----------------------------------------------------------------------------
# 5. CLINICAL ESCALATION RESPONSE
# Deterministic template. Never generated by an LLM. No hallucination risk.
# -----------------------------------------------------------------------------
CLINICAL_ESCALATION_MESSAGE = (
    "I understand you have a medical concern. For your safety, medical "
    "questions like this need to be reviewed by our clinical team.\n\n"
    "A licensed provider will respond within 2-4 hours during business hours. "
    "If your symptoms feel severe or urgent — especially dizziness, "
    "difficulty breathing, severe pain, or signs of allergic reaction — "
    "please call 911 or go to your nearest ER immediately.\n\n"
    "I've flagged your message for priority clinical review."
)


# -----------------------------------------------------------------------------
# 6. MAIN PIPELINE
# -----------------------------------------------------------------------------
def answer_with_routing(user_question: str) -> dict:
    """
    Full pipeline: classify → route → answer.
    Returns a dict with category, retrieved docs (if any), and the final answer.
    """
    category = classify_query(user_question)

    # Clinical: hard block. No RAG, no Sonnet.
    if category == "clinical":
        return {
            "category": category,
            "retrieved_docs": [],
            "answer": CLINICAL_ESCALATION_MESSAGE,
        }

    # Non-clinical: RAG + Sonnet with strict "context-only" guardrail
    retrieval = collection.query(query_texts=[user_question], n_results=2)
    retrieved_docs = retrieval["documents"][0]
    context = "\n\n".join(retrieved_docs)

    reply = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=300,
        system=(
            "You are MetaboLift's customer support assistant.\n"
            "Answer ONLY using the context below. If the answer is not in the "
            "context, say you don't know and recommend contacting "
            "support@metabolift.com. Keep responses short, warm, and professional.\n\n"
            f"CONTEXT:\n{context}"
        ),
        messages=[{"role": "user", "content": user_question}],
    )

    return {
        "category": category,
        "retrieved_docs": retrieval["ids"][0],
        "answer": reply.content[0].text,
    }


# -----------------------------------------------------------------------------
# 7. DEMO RUN
# -----------------------------------------------------------------------------
test_questions = [
    "When will my order arrive?",
    "Do you have a loyalty program with points?",
    "I'm having severe nausea and dizziness after my semaglutide shot. "
    "Should I stop taking it or lower my dose?",
]

for q in test_questions:
    print("=" * 72)
    print(f"QUESTION: {q}\n")
    result = answer_with_routing(q)
    print(f"Routing: {result['category']}")
    if result["retrieved_docs"]:
        print(f"Retrieved: {result['retrieved_docs']}")
    print(f"\nANSWER:\n{result['answer']}\n")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 73.2MiB/s]


QUESTION: When will my order arrive?

Routing: shipping
Retrieved: ['shipping', 'refunds']

ANSWER:
Your MetaboLift order will arrive **3-5 business days** after it ships via USPS Priority Mail. We ship all orders within 24 hours of approval, so you should receive it within about 4-6 business days total from when your order is approved.

You'll receive tracking information via email so you can monitor your delivery. All shipping is free! 📦

Is there anything else I can help you with regarding your order?

QUESTION: Do you have a loyalty program with points?

Routing: general
Retrieved: ['billing', 'refunds']

ANSWER:
I don't have information about a loyalty program or points system in my available resources. For questions about rewards or loyalty benefits, I'd recommend reaching out to our team at support@metabolift.com. They'll be happy to help with any program details!

QUESTION: I'm having severe nausea and dizziness after my semaglutide shot. Should I stop taking it or lower my dos